In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [ ]:
# Q21: Churn prediction feature dataset using Pandas feature engineering
# feature engineering is the process of creating new features or 
# modifying existing ones to improve the performance of machine learning models.
features = pd.DataFrame()

features['customerID']= df['customerID']
features['tenure'] = df['tenure']
features['monthly_charges'] = df['MonthlyCharges']
features['total_charges'] = df['TotalCharges']
features['clv'] = df['CLV']

features['is_senior'] = df['SeniorCitizen']
features['has_partner'] = (df['Partner'] == 'Yes').astype(int)
features['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)
features['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)
features['has_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)
features['has_techsupport'] = (df['TechSupport'] == 'Yes').astype(int)
features['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)

features['contract_monthly'] = (df['Contract'] == 'Month-to-month').astype(int)
features['contract_yearly']  = (df['Contract'] == 'One year').astype(int)

features['risk_score'] = features.apply(lambda row:
    (3 if df.loc[row.name, 'Contract'] == 'Month-to-month' else 1 if df.loc[row.name, 'Contract'] == 'One year' else 0) +
    (3 if row['tenure'] <= 12 else 2 if row['tenure'] <= 24 else 1 if row['tenure'] <= 48 else 0) +
    (2 if df.loc[row.name, 'TechSupport'] == 'No' else 0) +
    (2 if df.loc[row.name, 'OnlineSecurity'] == 'No' else 0), axis=1)

features['churn'] = (df['Churn'] == 'Yes').astype(int)

features.head()

,customerID,tenure,monthly_charges,total_charges,clv,is_senior,has_partner,has_dependents,has_phone,has_security,has_techsupport,is_paperless,contract_monthly,contract_yearly,risk_score,churn
0,7590-VHVEG,1,29.85,29.85,29.85,0,1,0,0,0,0,1,1,0,10,0
1,5575-GNVDE,34,56.95,1889.50,1936.30,0,0,0,1,1,0,0,0,1,4,0
2,3668-QPYBK,2,53.85,108.15,107.70,0,0,0,1,1,0,1,1,0,8,1
3,7795-CFOCW,45,42.30,1840.75,1903.50,0,0,0,0,1,1,0,0,1,2,0
4,9237-HQITU,2,70.70,151.65,141.40,0,0,0,1,0,0,1,1,0,10,1


In [ ]:
# Q23: Retention dashboard dataset with retention KPIs (Optional)

In [4]:
# Q24: Rule-based churn detection engine
def churn_risk_level(row):
    score = 0

    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1

    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1

    if row['OnlineSecurity'] == 'No':
        score += 2
    if row['TechSupport'] == 'No':
        score += 2

    if row['PaymentMethod'] == 'Electronic check':
        score += 1

    if row['SeniorCitizen'] == 1:
        score += 1

    if score >= 9:
        return 'Critical'
    elif score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'

df['churn_risk'] = df.apply(churn_risk_level, axis=1)

df.groupby('churn_risk').agg(
    customer_count = ('customerID', 'count'),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index().sort_values('churn_rate', ascending=False)

,churn_risk,customer_count,churn_rate
0,Critical,2105,56.72
1,High,1764,27.61
3,Medium,1377,9.37
2,Low,1797,3.28
